## Linkedin Ads


#### Useful links:

- [LinkedIn Ad Library API Program](https://www.linkedin.com/ad-library/api)
- [Linkedin Developer Portal](https://www.linkedin.com/developers/)
- [Linkedin Ads API Schema](https://www.linkedin.com/ad-library/api/ads)
- [Linkedin Ad Library Interface](https://www.linkedin.com/ad-library/)


In [143]:
# Requirements
# %pip install pandas requests python-dotenv

In [1]:
import json
import os
import time
from datetime import datetime, timezone
from typing import Any

import pandas as pd
import requests
from dateutil.relativedelta import relativedelta
from dotenv import load_dotenv
from pydantic import BaseModel
from rich import print as pprint

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)
pd.set_option("display.width", 1000)

DATA_DIR = "../../data"
BASE_NAME = "br-linkedin-ads"


# File management
def save_json(question_info: str, json_data: list[Any] | dict[str, Any]):
    now_str = datetime.now(tz=timezone.utc).isoformat()
    filepath = f"{DATA_DIR}/{BASE_NAME}-{question_info}-{now_str}.json"
    print(f"----> {filepath}")
    with open(filepath, "w") as fout:
        json.dump(json_data, fout, indent=4, ensure_ascii=False)


# Environment variables
def get_headers():
    load_dotenv()
    return {
        "X-RestLi-Protocol-Version": "2.0.0",
        "LinkedIn-Version": "202503",
        "Authorization": f"Bearer {os.getenv('LINKEDIN_ACCESS_TOKEN')}",
    }

#### API Connection


In [2]:
ENDPOINT: str = "https://api.linkedin.com/rest/adLibrary"


class Countries(BaseModel):
    codes: list[str]

    def __str__(self) -> str:
        urns = ",".join([f"urn:li:country:{code}" for code in self.codes])
        return f"(value:List({urns.replace(':', '%3A')}))"


def make_datetime(obj: str | datetime | None) -> datetime | None:
    if isinstance(obj, str):
        obj = datetime.fromisoformat(obj)
    if isinstance(obj, datetime):
        return obj.astimezone(tz=timezone.utc)


class DateRange(BaseModel):
    start: datetime
    end: datetime

    def __init__(
        self,
        start: str | datetime | None = None,
        end: str | datetime | None = None,
    ) -> None:
        now = datetime.now(tz=timezone.utc)
        start = make_datetime(start) or (now - relativedelta(years=1))
        end = make_datetime(end) or (now - relativedelta(days=1))
        super().__init__(start=start, end=end)

    def __str__(self) -> str:
        """Maximum range: `Between yesterday and today minus 1 year.`"""
        range_start = self.start.strftime(r"(start:(day:%d,month:%m,year:%Y)")
        range_end = self.end.strftime(r"end:(day:%d,month:%m,year:%Y))")
        return ",".join((range_start, range_end))


class Params(BaseModel):
    dateRange: DateRange = DateRange()
    countries: Countries | list[str] | None = None
    advertiser: str | None = None
    keyword: str | None = None
    start: int | str = 0
    count: int | str = 25

    def model_post_init(self, context: Any) -> None:
        if isinstance(codes := self.countries, list):
            self.countries = Countries(codes=codes)
        return super().model_post_init(context)

    @property
    def url(self) -> str:
        items = {"q": "criteria", **self.__dict__}.items()
        items = [f"{k}={v}" for k, v in items if v is not None]
        return ENDPOINT.strip("/") + "?" + "&".join(items).replace(" ", "%20")


def get_ads(
    params: Params | None = None,
    api_url: str | None = None,
    save_as: str | None = None,
    show_df: bool = True,
    sleep: int | float = 1,
) -> Any | None:
    api_url = api_url or (params or Params()).url
    time.sleep(sleep)
    print(f"[GET] {api_url}")
    resp = requests.get(api_url, headers=get_headers(), timeout=30)
    data = resp.json() if resp and resp.ok else json.loads(resp.text)
    data = data if isinstance(data, dict) else {"elements": data}
    if "elements" not in data:
        return pprint(data)
    if save_as:
        save_json(save_as, data)
    if show_df:
        display(pd.json_normalize(data.get("elements", []))[:3].T)
    return data

### Special Criteria


**SC1: Does the platform provide an API to access its ad repository and extract data on advertising content?**

_This item verifies whether the platform provides an ad repository API with at least one endpoint for programmatically extracting advertising data. Full availability is confirmed when the API returns information on ads across all categories. The assessment should confirm that the endpoint allows the retrieval and storage of ad data without requiring privileged or internal access beyond standard developer registration._


In [3]:
# Yes (for API). Requesting ads from Brazil (BR) with "Petroleo" keyword:
data = get_ads(
    Params(countries=["BR"], keyword="Petroleo"), save_as="question-01-data"
)

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&keyword=Petroleo&start=0&count=25
----> ../../data/br-linkedin-ads-question-01-data-2025-11-10T16:06:45.807210+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/918...,https://www.linkedin.com/ad-library/detail/918...,https://www.linkedin.com/ad-library/detail/946...
details.type,SPONSORED_VIDEO,SPONSORED_STATUS_UPDATE,SPONSORED_VIDEO
details.advertiser.adPayer,MJB Editores Associados Ltda,INSTITUTO BRASILEIRO DE PETROLEO E GAS,AGGREKO UK LIMITED
details.advertiser.advertiserName,Leandro Villela,"IBP - Instituto Brasileiro de Petróleo, Gás e ...",Aggreko
details.advertiser.advertiserUrl,https://www.linkedin.com/in/leandro-villela-12...,https://www.linkedin.com/company/438044,https://www.linkedin.com/company/13151
details.adTargeting,[],[],[]
details.adStatistics.latestImpressionAt,NaN,NaN,NaN
details.adStatistics.impressionsDistributionByCountry,NaN,NaN,NaN
details.adStatistics.totalImpressions.from,NaN,NaN,NaN


**SC3: Can data from both active and inactive ads be extracted?**

_This item verifies whether the platform allows the extraction of ad data through either the GUI or the API, from at least one day after publication to at least one year prior. Full availability is confirmed when both active and inactive ad data are delivered across all ad categories. The assessment should test the interface and endpoints to confirm whether both active and inactive ads can be retrieved._


* **No**. The API only offers the information if the ad is restricted or not, through the field `isRestricted`. From the API documentation, it is a "boolean explaining if the ad was removed for violating LinkedIn’s Advertising Policy.". In terms of inactive ads, it is possible to obtain data from ads with `latestImpressionAt` in the past, but that information is only available at the response schema for EU countries.
* Refs.:
    - https://www.linkedin.com/ad-library/api/ads
    - https://www.linkedin.com/help/lms/answer/a1620070

### ACCESSIBILITY


**OC3: Can the requested data be extracted directly from the ad repository response?**

This item verifies whether the platform ad repository returns structured data on ad creatives and authorship directly in the response, rather than providing a link that redirects to the data. Audiovisual media files and data (e.g., images, videos, and audio) should not be considered when assessing this item. The assessment should examine sample data responses from both the ad repository GUI and API to confirm that the requested public data is included in the returned payload.\*


* **No** (for API). The endpoint returns only the ad's URL and data about the advertiser, including `advertiserName`, `advertiserUrl` and `adPayer`, but neither the ad's text nor publication date.

**OC5: Can data from an individual ad be retrieved from the platform?**

_This item verifies whether it is possible to retrieve data from a specific advertisement on the ad repository using a unique identifier, rather than relying on search terms or other parameters and filters. The assessment should review the ad repository documentation and test available features to confirm that an individual ad can be retrieved directly by its unique identifier._


* **No** (for API). There is no endpoint for that. To get data from an individual ad it must be requested outside the API via browser or HTTP request.

**OC6: Can data from ads served by a specific advertiser be retrieved from the platform?**

_This item verifies whether it is possible to retrieve data from ads run by a specific advertiser, via their username or unique identifier. The assessment should review the ad repository documentation and test any available feature to retrieve data from an individual advertiser._


In [4]:
# No (for API). The search for advertisers only retrieves data that contains the
# searched keyword instead of filtering data from a specific advertiser via
# their username or unique identifier.
data = get_ads(
    Params(countries=["BR"], advertiser="Shell"), save_as="question-06-data"
)


[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&advertiser=Shell&start=0&count=25
----> ../../data/br-linkedin-ads-question-06-data-2025-11-10T16:06:57.526275+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/880...,https://www.linkedin.com/ad-library/detail/895...,https://www.linkedin.com/ad-library/detail/880...
details.type,SPONSORED_STATUS_UPDATE,SPONSORED_STATUS_UPDATE,SPONSORED_STATUS_UPDATE
details.advertiser.adPayer,Shell Chemical LP,Shell Brasil Petróleo Ltda.,Shell Chemical LP
details.advertiser.advertiserName,Shell Chemicals,Shell,Shell Chemicals
details.advertiser.advertiserUrl,https://www.linkedin.com/company/99846396,https://www.linkedin.com/company/1271,https://www.linkedin.com/company/99846396
details.adTargeting,[],[],[]
details.adStatistics.latestImpressionAt,NaN,NaN,NaN
details.adStatistics.impressionsDistributionByCountry,NaN,NaN,NaN
details.adStatistics.totalImpressions.from,NaN,NaN,NaN


**OC7: Can ad data be retrieved from the platform using search terms?**

_This item verifies whether data on ad campaigns can be retrieved via individual or combined search terms, enabling the creation of a dataset composed of ads that mention those terms. The assessment should test search-related features to confirm that queries using keywords return relevant ad campaign data._


In [5]:
# Yes. We just need to pass the parameter "keyword":
data = get_ads(
    Params(countries=["BR"], keyword="Petroleo"), save_as="question-07-data"
)

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&keyword=Petroleo&start=0&count=25
----> ../../data/br-linkedin-ads-question-07-data-2025-11-10T16:07:09.392788+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/918...,https://www.linkedin.com/ad-library/detail/918...,https://www.linkedin.com/ad-library/detail/946...
details.type,SPONSORED_VIDEO,SPONSORED_STATUS_UPDATE,SPONSORED_VIDEO
details.advertiser.adPayer,MJB Editores Associados Ltda,INSTITUTO BRASILEIRO DE PETROLEO E GAS,AGGREKO UK LIMITED
details.advertiser.advertiserName,Leandro Villela,"IBP - Instituto Brasileiro de Petróleo, Gás e ...",Aggreko
details.advertiser.advertiserUrl,https://www.linkedin.com/in/leandro-villela-12...,https://www.linkedin.com/company/438044,https://www.linkedin.com/company/13151
details.adTargeting,[],[],[]
details.adStatistics.latestImpressionAt,NaN,NaN,NaN
details.adStatistics.impressionsDistributionByCountry,NaN,NaN,NaN
details.adStatistics.totalImpressions.from,NaN,NaN,NaN


**OC8: Does the platform use locale-neutral data representations?**

_This item verifies whether locale-sensitive data (e.g., timestamps, currency, numbers) are provided in a locale-neutral format, or, if that is not possible, whether relevant locale metadata is included. The assessment should review the ad repository documentation and inspect sample responses to confirm the presence of standardized formats or accompanying metadata._


* **No**. From the documentation: "All dates and timestamps are recorded in Coordinated Universal Time (UTC)". However, it cannot be demonstrated because time-related fields are only available for EU countries.
* Refs.:
    - https://www.linkedin.com/ad-library
    - https://www.linkedin.com/help/lms/answer/a1620070

### COMPLETENESS


**OC9: Does the platform provide data that allows the identification of advertisers who ran ads?**

_This item verifies whether the platform discloses information on the advertisers responsible for the identified ads. The assessment should confirm whether the advertiser’s page name, URL, and unique identifier can be retrieved._


In [6]:
# Yes.
data = get_ads(
    Params(countries=["BR"], keyword="Petroleo"),
    save_as="question-09-data",
    show_df=False,
)
# Showing details about advertisers.
if data:
    display(pd.json_normalize(data["elements"]).filter(like="advertiser"))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&keyword=Petroleo&start=0&count=25
----> ../../data/br-linkedin-ads-question-09-data-2025-11-10T16:07:30.166230+00:00.json


,details.advertiser.adPayer,details.advertiser.advertiserName,details.advertiser.advertiserUrl
0,MJB Editores Associados Ltda,Leandro Villela,https://www.linkedin.com/in/leandro-villela-12...
1,INSTITUTO BRASILEIRO DE PETROLEO E GAS,"IBP - Instituto Brasileiro de Petróleo, Gás e ...",https://www.linkedin.com/company/438044
2,AGGREKO UK LIMITED,Aggreko,https://www.linkedin.com/company/13151
3,Elite Training,Elite Training,https://www.linkedin.com/company/2732226
4,Túlio Canhoni,Túlio Canhoni,https://www.linkedin.com/in/tcanhoni
5,PETROLEO BRASILEIRO S A PETROBRAS,Petrobras,https://www.linkedin.com/company/4120
6,Bruno Bellato,ParCerto Consultoria de Vendas,https://www.linkedin.com/company/108795087
7,Octans Conteúdo Digital,Vinícius Macedo Silva,https://www.linkedin.com/in/vinicius-macedo-silva
8,Douglas Amado Lagame,"Douglas Lagame, PMP",https://www.linkedin.com/in/douglas-lagame
9,INSTITUTO BRASILEIRO DE PETROLEO E GAS,UnIBP - A Universidade Setorial do Instituto B...,https://www.linkedin.com/company/82702424


**OC10: Does the platform provide data on the funders who paid for ads?**

_This item verifies whether the platform provides data on who paid for ad campaigns. The assessment should confirm whether any sponsor information can be retrieved._


In [7]:
# Yes
data = get_ads(
    Params(countries=["BR"], keyword="Petroleo"),
    save_as="question-10-data",
    show_df=False,
)
# Showing only the adPayer field.
if data:
    display(pd.json_normalize(data["elements"]).filter(like="adPayer"))

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&keyword=Petroleo&start=0&count=25
----> ../../data/br-linkedin-ads-question-10-data-2025-11-10T16:07:32.569209+00:00.json


,details.advertiser.adPayer
0,MJB Editores Associados Ltda
1,INSTITUTO BRASILEIRO DE PETROLEO E GAS
2,AGGREKO UK LIMITED
3,Elite Training
4,Túlio Canhoni
5,PETROLEO BRASILEIRO S A PETROBRAS
6,Bruno Bellato
7,Octans Conteúdo Digital
8,Douglas Amado Lagame
9,INSTITUTO BRASILEIRO DE PETROLEO E GAS


**OC11: Does the platform provide data on the period during which ads were served?**

_This item verifies whether the platform provides data on the days on which ad campaigns ran. The assessment should review ad records to confirm that campaigns include start and end dates (or equivalent temporal markers) covering their active period_


* **No**. The response schema shows `firstImpressionAt` and `latestImpressionAt` only for EU countries.
* Ref.: https://www.linkedin.com/help/lms/answer/a1620070

**OC12: Does the platform provide data on user engagement with ads?**

_This item verifies whether the platform provides data on the total number of user interactions with ad campaigns (e.g., likes, comments, shares, clicks). The assessment should review ad records to confirm that engagement metrics are available and clearly linked to each campaign._


* **No**. This is available only for EU countries.
* Ref.: https://www.linkedin.com/help/lms/answer/a1620070

In [8]:
# Showing data for Germany (an EU country):
data = get_ads(Params(countries=["DE"]), save_as="question-12-data")

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ADE))&start=0&count=25
----> ../../data/br-linkedin-ads-question-12-data-2025-11-10T16:07:35.439002+00:00.json


,0,1,2
isRestricted,False,False,False
adUrl,https://www.linkedin.com/ad-library/detail/895...,https://www.linkedin.com/ad-library/detail/957...,https://www.linkedin.com/ad-library/detail/954...
details.advertiser.adPayer,The Fairly Good Company Ltd,"Pantheon Systems, Inc.","Pantheon Systems, Inc."
details.advertiser.advertiserName,Revenu,Pantheon,Pantheon
details.advertiser.advertiserUrl,https://www.linkedin.com/company/88963744,https://www.linkedin.com/company/2198587,https://www.linkedin.com/company/2198587
details.adStatistics.latestImpressionAt,1762645409681,1762646313136,1762645666689
details.adStatistics.impressionsDistributionByCountry,[],[],[]
details.adStatistics.totalImpressions.from,0,0,0
details.adStatistics.totalImpressions.to,1000,1000,1000
details.adStatistics.firstImpressionAt,1762642915846,1762638805531,1762639077844


**OC13: Does the platform indicate whether ads were placed by verified or unverified advertisers?**

_This item verifies whether the platform clearly indicates whether advertisers were verified at the time their ads were served. The assessment should review ad records to confirm that a verification status field is present._


* **No**. The only fields about advertisers in response schema are: `advertiserName`, `advertiserUrl` and `adPayer`.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### COMPLIANCE


**OC14: Does the platform flag ads that were removed due to violations of its guidelines or relevant legislation?**

_This item verifies whether the platform indicates when an ad has been moderated. At a minimum, the platform should provide the reason for removal and the date. The assessment should review ad records to confirm that moderated ads are flagged and that the corresponding moderation details are clearly documented._


* **No**. Even though the very documentation's Sample Response shows a case with `isRestricted: True`, the API seems to return only ads with `isRestricted: False` and it is not possible to search for restricted ads.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC15: Does the platform indicate whether ad creatives were generated using artificial intelligence?**

_This item verifies whether the platform flags ad campaigns in which AI played a role in producing the content. The assessment should review ad records to confirm the presence of a field or label indicating the use of AI in ad production._


* **No**. There are no mentions to AI anywhere.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### CONSISTENCY


**OC23: Does the data retrieved by the API reflect what is displayed on the platform’s ad repository GUI?**

_This item verifies whether the data returned by the platform’s ad repository API corresponds to the information displayed on its GUI in all its levels of detail. It should be possible to identify in the API response information such as authorship, complete content, and other campaigning information (e.g., amount spent, impressions reached). The assessment should compare API responses with the GUI to confirm that key elements—such as authorship, full content, and campaign information (e.g., spending, impressions)—are consistently included._


* **No**.
* Although the data retrieved by the API is similar to what is displayed on the GUI, the platform does not offer enough information and manners to properly  compare. The only relevant fields are about advertisers in response schema: advertiserName, advertiserUrl and adPayer, which are insufficient to compare data from API and GUI based on the OC's requirements.
* The GUI search page shows a preview of each ad's media and text, while the API returns no information whatsoever about each ad's content, only its URL and advertiser data.

**OC24: Are the results returned by the platform consistently reproducible?**

_This item verifies whether data accessed and extracted via the platform’s ad repository at a given time is consistent with other collections performed similarly, including cases where content was deleted in the interim. The assessment should perform repeated queries to confirm reproducibility of results._


Test instructions: Please, develop a test that collects ads for 5 times using the same request parameters and the same endpoint. Save each response in separate files.


In [9]:
total_runs = 5
responses = []
for index in range(1, total_runs + 1):
    resp = get_ads(
        Params(countries=["BR"], advertiser="Petrobras"),
        save_as=f"question-24-run-{index}",
        show_df=False,
        sleep=7,
    )
    responses.append(resp)
comparison = (
    responses[0] == responses[1] == responses[2] == responses[3] == responses[4]
)
pprint(f"All equal: {comparison}")

[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&advertiser=Petrobras&start=0&count=25
----> ../../data/br-linkedin-ads-question-24-run-1-2025-11-10T16:07:43.581023+00:00.json
[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&advertiser=Petrobras&start=0&count=25
----> ../../data/br-linkedin-ads-question-24-run-2-2025-11-10T16:07:51.669583+00:00.json
[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(start:(day:10,month:11,year:2024),end:(day:09,month:11,year:2025))&countries=(value:List(urn%3Ali%3Acountry%3ABR))&advertiser=Petrobras&start=0&count=25
----> ../../data/br-linkedin-ads-question-24-run-3-2025-11-10T16:07:59.700586+00:00.json
[GET] https://api.linkedin.com/rest/adLibrary?q=criteria&dateRange=(st

All equal: True

**OC25: Is the data returned by the platform consistent with the parameters and filters used in the request?**

_This item verifies whether the data retrieved through the ad repository accurately reflects the parameters and filters specified at the time of the request. The assessment should run test queries with different filters to confirm that results consistently match the requested conditions._


Test instructions: Please, develop a process that request data using different parameter to the same endpoint. If possible, test at least 5 different parameters/filters. If the platform provides less than 5, use all available parameters/filters.

Save each response in separate files.


* **No**. The information cannot be verified through the API response alone. Only the advertiser details are returned at the API response for countries outside the EU, so it is the only information that could be verified programatically without scraping the ad page itself with the `adUrl` provided.
* Refs.:
    - https://www.linkedin.com/ad-library/api/ads
    - https://www.linkedin.com/help/lms/answer/a1620070

### RELEVANCE


**OC26: Does the platform allow the use of temporal filters to retrieve data on ads?**

_This item verifies whether the ad repository allows filtering ad campaign data based on the time period during which the ads were served. The assessment should test queries with temporal filters to confirm that results accurately reflect the specified date ranges._


* **No**. The dateRange filter exists but the API's response cannot be tested for it because the response schema shows `firstImpressionAt` and `latestImpressionAt` only for EU countries.
* Ref.: https://www.linkedin.com/help/lms/answer/a1620070

**OC27: Does the platform allow filtering advertising data by ad category?**

_This item verifies whether the ad repository allows filtering ad campaign data based on any possible categories assigned at the time of ad creation. The assessment should run test queries with category filters to confirm that results align with the selected classifications._


* **No**. There is no category parameter at the request schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC28: Does the platform allow filtering advertising data by geographic location?**

_This item assesses whether the ad repository allows filtering data by one or more subnational geographic locations where the ads were served. The assessment should test queries with location filters to confirm that results match the specified areas._


* **No**. There is no information about subnational locations at the API.
* Ref.: https://www.linkedin.com/ad-library/api/ads

### ACCURACY


**OC29: Does the platform provide age and gender data on the audiences of ads?**

_This item verifies whether the platform provides data on the age and gender of audiences reached. The assessment should review the ad records to confirm that these breakdowns are available and consistently reported_


* **No**. There is neither age nor gender parameters at the request schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC30: Does the platform provide subnational geographic data on the audience reached by ads?**

_This item verifies whether the platform provides data on the subnational geographic location of audiences reached. The assessment should review the ad records to confirm that such location breakdowns are available and consistently reported._


* **No**. There is no subnational option at the response schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads

**OC31: Does the platform include data on audience targeting criteria defined by advertisers?**

_This item verifies whether the platform provides data on audience targeting criteria defined by the advertiser when publishing ads (e.g., demographic and geographic segments, as well as interests, attitudes, behaviors, and keywords). The assessment should review ad records to confirm that these targeting parameters are available and consistently reported._


* **No**. This feature is only available for EU countries.
* Ref.: https://www.linkedin.com/help/lms/answer/a1620070

**OC32: Does the platform provide granular volume ranges for ad impressions?**

_This item verifies whether the ad repository provides impression values for ads, using ranges that closely approximate the actual numbers. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to 1,000 impressions should be displayed in intervals no larger than 100; between 1,000 and 10,000 in intervals no larger than 1,000; between 10,000 and 100,000 in intervals no larger than 10,000; between 100,000 and 1 million or above, in intervals no larger than 100,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces._


* **No**. This feature is only available for EU countries.
* Ref.: https://www.linkedin.com/help/lms/answer/a1620070

**OC33: Does the platform provide granular investment ranges for ad spending?**

_This item verifies whether the ad repository provides spending values for ads, using ranges that closely approximate the actual amounts. Intervals should be no larger than 10% of the upper bound of the value range they represent. For example, values up to $100 should be displayed in intervals no larger than $10; between $100 and $1,000 in intervals no larger than $100; and between $1,000 and $10,000 in intervals no larger than $1,000. The assessment should measure whether the reported intervals remain within this threshold across the different value ranges using the platform’s documentation or available data interfaces._


* **No**. There is no ad spending informations at the response schema.
* Ref.: https://www.linkedin.com/ad-library/api/ads